# Phase 2.1 — TrustFin-SLM Research Validation and Phase-3/4 Data Contract
## Research: *Trustworthy Small Language Models for Financial Tasks*

This notebook is a **research-ready revision of the successfully executed Phase-2 notebook**. It preserves the Phase-2 data contract and benchmark-isolation philosophy, while applying four corrections before final modeling:

1. **Replace unavailable FinNLI with FiQA sentiment** using FiQA's native continuous score in \([-1,1]\).
2. **Keep FinBen-FOMC strictly test-only** as an external financial generalization benchmark.
3. **Validate every FinQA gold program with the official FinQA executor**, rather than relying on `steps[-1].res`.
4. **Make research-integrity checks mandatory**: exact leakage, near-duplicate audit, protected-test isolation, source/artifact fingerprints, and final PASS/FAIL gates.

### Downstream policy

- **Phase-3 training / validation:** FinQA + FiQA only.
- **Protected evaluation:** FinQA test + FiQA test + FinBen-FOMC + FinTrust.
- **FinTrust never enters SFT, calibration, threshold selection, or prompt-example pools.**
- **FinBen-FOMC never enters SFT or threshold selection**, because the public benchmark exposes only a test split.
- **All downstream notebooks should read the exported registry/config files rather than reloading raw datasets independently.**

The notebook is designed to run standalone, but if the earlier Phase-2 registry exists under the same workspace it records that provenance in the Phase-2.1 manifest.

## 0. Environment and reproducibility

The dependency cell installs only missing packages. The notebook intentionally uses deterministic serialization and fixed seeds for auditability.

In [ ]:
from __future__ import annotations

import importlib.util
import sys
import subprocess

_REQUIRED = {
    "pandas": "pandas>=2.0",
    "numpy": "numpy>=1.24",
    "requests": "requests>=2.31",
    "datasets": "datasets>=3.0",
    "huggingface_hub": "huggingface_hub>=0.25",
    "pyarrow": "pyarrow>=15",
    "sklearn": "scikit-learn>=1.4",
    "tqdm": "tqdm>=4.66",
    "datasketch": "datasketch>=1.6",
    "sympy": "sympy>=1.12",
}
_missing = [pkg for mod, pkg in _REQUIRED.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print("Installing missing packages:", _missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])
else:
    print("All required packages are already available.")

In [ ]:
import csv
import hashlib
import json
import os
import platform
import random
import re
import shutil
import time
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

## 1. Configuration

The workspace convention is inherited from Phase-2. Set `TRUSTFIN_ROOT` if you want an explicit location. Otherwise, the notebook creates/uses the current working directory (`.`).

Phase-2.1 writes to a **new artifact directory** and never overwrites Phase-2 exports.

In [ ]:
PROJECT_ROOT = Path(os.environ.get("TRUSTFIN_ROOT", ".")).resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"

PHASE2_ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "phase2"
PHASE2_REGISTRY_PATH = PHASE2_ARTIFACT_DIR / "manifests" / "dataset_registry.json"

ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "phase2_1"
EXPORT_DIR = ARTIFACT_DIR / "datasets"
EVAL_DIR = ARTIFACT_DIR / "eval"
MANIFEST_DIR = ARTIFACT_DIR / "manifests"
MODULE_DIR = ARTIFACT_DIR / "python"
AUDIT_DIR = ARTIFACT_DIR / "audits"

for p in [RAW_DIR, EXPORT_DIR, EVAL_DIR, MANIFEST_DIR, MODULE_DIR, AUDIT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# FinQA canonical release + official evaluator.
FINQA_BASE_URL = "https://raw.githubusercontent.com/czyssrs/FinQA/main/dataset"
FINQA_EVALUATOR_URL = "https://raw.githubusercontent.com/czyssrs/FinQA/main/code/evaluate/evaluate.py"
FINQA_EXPECTED = {"train": 6251, "dev": 883, "test": 1147}

# FiQA Task-1 style aspect-aware sentiment score prediction.
FIQA_HF_REPO = "TheFinAI/fiqa-sentiment-classification"
FIQA_SPLIT_MAP = {"train": "train", "valid": "valid", "test": "test"}
FIQA_EXPECTED = {"train": 822, "valid": 117, "test": 234}

# FinBen-FOMC is protected external evaluation only.
FINBEN_FOMC_HF_REPO = "TheFinAI/finben-fomc"
FINBEN_FOMC_SPLIT = "test"
FINBEN_FOMC_EXPECTED = 496
FINBEN_FOMC_LABELS = ["dovish", "hawkish", "neutral"]

# Optional HF token; public datasets normally do not require it.
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()

# FinTrust is protected trust evaluation only.
FINTRUST_HF_REPO = "HughieHu/FinTrust"
FINTRUST_GITHUB = "https://github.com/HughieHu/FinTrust.git"
FINTRUST_DIMENSIONS = [
    "trustfulness", "robustness", "safety", "fairness", "privacy",
    "transparency", "knowledge_discovery",
]

# Integrity settings.
DOWNLOAD_TIMEOUT = 90
RUN_NEAR_DUP_AUDIT = True
NEAR_DUP_THRESHOLD = 0.90
MINHASH_PERMUTATIONS = 96
WORD_SHINGLE_N = 3

SCHEMA_VERSION = "trustfin-phase2.1-v1.0"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Phase-2 registry exists:", PHASE2_REGISTRY_PATH.exists())
print("Phase-2.1 artifacts:", ARTIFACT_DIR)
print("HF_TOKEN configured:", bool(HF_TOKEN))

### Phase-2 provenance bridge

If the successfully executed Phase-2 registry exists, Phase-2.1 records its SHA-256 hash. This provides an explicit lineage from Phase-2 to Phase-2.1 without making Phase-2.1 dependent on stale exported records.

In [ ]:
phase2_parent_provenance = None
if PHASE2_REGISTRY_PATH.exists():
    phase2_parent_provenance = {
        "registry_path": str(PHASE2_REGISTRY_PATH.resolve()),
        "sha256": hashlib.sha256(PHASE2_REGISTRY_PATH.read_bytes()).hexdigest(),
    }
    print("Parent Phase-2 registry:", phase2_parent_provenance)
else:
    print("Parent Phase-2 registry not found; Phase-2.1 will run standalone.")

## 2. Shared utilities and validation gates

In [ ]:
def json_default(x: Any):
    if isinstance(x, (np.integer,)): return int(x)
    if isinstance(x, (np.floating,)): return float(x)
    if isinstance(x, np.ndarray): return x.tolist()
    if isinstance(x, Path): return str(x)
    if pd.isna(x) if not isinstance(x, (list, dict, tuple, set)) else False: return None
    return str(x)


def canonical_json(obj: Any) -> str:
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"), default=json_default)


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def normalized_text(text: Any) -> str:
    s = "" if text is None else str(text)
    s = s.lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s


def input_signature(*parts: Any) -> str:
    return sha256_text("\n<SEP>\n".join(normalized_text(x) for x in parts))


def write_jsonl(records: Iterable[Mapping[str, Any]], path: Path) -> int:
    path.parent.mkdir(parents=True, exist_ok=True)
    n = 0
    with path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(dict(r), ensure_ascii=False, default=json_default) + "\n")
            n += 1
    return n


def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    out = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                out.append(json.loads(line))
    return out


def save_json(obj: Any, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=json_default), encoding="utf-8")


def load_json_or_jsonl(path: Path) -> List[Dict[str, Any]]:
    text = path.read_text(encoding="utf-8-sig").strip()
    if not text:
        return []
    try:
        obj = json.loads(text)
        if isinstance(obj, list):
            return [x for x in obj if isinstance(x, dict)]
        if isinstance(obj, dict):
            # Common wrappers.
            for key in ["data", "records", "examples", "items", "instances"]:
                if isinstance(obj.get(key), list):
                    return [x for x in obj[key] if isinstance(x, dict)]
            return [obj]
    except json.JSONDecodeError:
        pass
    rows = []
    for line in text.splitlines():
        line = line.strip()
        if line:
            rows.append(json.loads(line))
    return [x for x in rows if isinstance(x, dict)]


def load_tabular_file(path: Path) -> pd.DataFrame:
    ext = path.suffix.lower()
    if ext in {".json", ".jsonl"}:
        return pd.DataFrame(load_json_or_jsonl(path))
    if ext == ".csv":
        return pd.read_csv(path)
    if ext == ".tsv":
        return pd.read_csv(path, sep="\t")
    if ext == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported file: {path}")


@dataclass
class Gate:
    name: str
    passed: bool
    observed: Any = None
    expected: Any = None
    severity: str = "ERROR"
    note: str = ""

GATES: List[Gate] = []

def add_gate(name: str, passed: bool, observed=None, expected=None, severity="ERROR", note=""):
    g = Gate(name, bool(passed), observed, expected, severity, note)
    GATES.append(g)
    status = "PASS" if g.passed else ("WARN" if severity == "WARNING" else "FAIL")
    print(f"[{status}] {name}")
    if not g.passed and note:
        print("       ", note)
    return g

In [ ]:
def download_file(url: str, path: Path, overwrite: bool = False) -> Path:
    if path.exists() and not overwrite:
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    r = requests.get(url, timeout=DOWNLOAD_TIMEOUT)
    r.raise_for_status()
    path.write_bytes(r.content)
    return path


def hf_dataset_to_pandas(repo_id: str, split: str) -> pd.DataFrame:
    from datasets import load_dataset
    ds = load_dataset(repo_id, split=split, token=HF_TOKEN or None)
    return ds.to_pandas()


def safe_float(x: Any) -> Optional[float]:
    try:
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return None
        return float(x)
    except Exception:
        return None


def assert_record_contract(records: Sequence[Mapping[str, Any]], name: str):
    required = {
        "record_id", "source_id", "dataset", "task", "split", "system",
        "instruction", "input", "target", "target_answer", "target_label",
        "gold_program", "gold_evidence", "metadata",
    }
    missing = []
    for i, r in enumerate(records):
        diff = required - set(r.keys())
        if diff:
            missing.append((i, sorted(diff)))
            if len(missing) >= 10:
                break
    add_gate(f"{name}: canonical record schema", len(missing) == 0, missing, [])

# Part A — FinQA
## 3. Load canonical FinQA splits

Phase-2.1 preserves the Phase-2 FinQA loading strategy. Existing raw files are reused; missing files are downloaded from the official repository.

In [ ]:
FINQA_RAW_DIR = RAW_DIR / "finqa"
FINQA_RAW_DIR.mkdir(parents=True, exist_ok=True)

finqa_raw_paths = {}
for split, filename in {"train": "train.json", "dev": "dev.json", "test": "test.json"}.items():
    p = FINQA_RAW_DIR / filename
    download_file(f"{FINQA_BASE_URL}/{filename}", p)
    finqa_raw_paths[split] = p

finqa_source_manifest = {
    split: {
        "path": str(path.resolve()),
        "sha256": sha256_file(path),
        "bytes": path.stat().st_size,
    }
    for split, path in finqa_raw_paths.items()
}
display(pd.DataFrame(finqa_source_manifest).T)

In [ ]:
def finqa_table_to_text(table: Any) -> str:
    if not isinstance(table, list):
        return ""
    return "\n".join(" | ".join(map(str, row)) for row in table if isinstance(row, list))


def finqa_gold_evidence_to_text(gold_inds: Any) -> str:
    if isinstance(gold_inds, dict):
        return "\n".join(str(v) for _, v in sorted(gold_inds.items()))
    if isinstance(gold_inds, list):
        return "\n".join(map(str, gold_inds))
    return ""


def flatten_finqa(raw_rows: Sequence[Dict[str, Any]], split: str) -> pd.DataFrame:
    rows = []
    for item in raw_rows:
        qa = item.get("qa", {}) if isinstance(item.get("qa"), dict) else {}
        steps = qa.get("steps", []) or []
        final_step_result = None
        if steps and isinstance(steps[-1], dict):
            final_step_result = steps[-1].get("res")

        pre = item.get("pre_text", []) or []
        post = item.get("post_text", []) or []
        table = item.get("table", []) or []
        gold_inds = qa.get("gold_inds", {}) or {}

        rows.append({
            "id": str(item.get("id")),
            "split": split,
            "pre_text": pre,
            "post_text": post,
            "table": table,
            "question": qa.get("question"),
            "answer": qa.get("answer"),
            "exe_ans": qa.get("exe_ans"),
            "program": qa.get("program"),
            "program_re": qa.get("program_re"),
            "steps": steps,
            "gold_inds": gold_inds,
            "final_step_result": final_step_result,
            "context_text": "\n".join(map(str, pre + post)).strip(),
            "table_text": finqa_table_to_text(table),
            "gold_evidence_text": finqa_gold_evidence_to_text(gold_inds),
        })
    return pd.DataFrame(rows)

finqa_splits = {}
for split, path in finqa_raw_paths.items():
    raw = json.loads(path.read_text(encoding="utf-8"))
    finqa_splits[split] = flatten_finqa(raw, split)

finqa_df = pd.concat(finqa_splits.values(), ignore_index=True)
print(finqa_df.groupby("split").size())
display(finqa_df.head(2))

## 4. FinQA structural integrity

The old `steps[-1].res` diagnostic is retained only as **annotation metadata**. It is not used as the research-valid execution check. The decisive check appears in the next section and uses the official FinQA evaluator.

In [ ]:
#in this section the    
def parse_number_loose(x: Any) -> Optional[float]:
    if x is None: return None
    s = str(x).strip().lower().replace(",", "").replace("$", "")
    if not s or s in {"none", "nan", "n/a"}: return None
    is_percent = s.endswith("%")
    s = s[:-1].strip() if is_percent else s
    try:
        v = float(s)
        return v / 100.0 if is_percent else v
    except ValueError:
        return None


def loose_equivalent(a: Any, b: Any, atol=1e-4, rtol=1e-3) -> bool:
    sa, sb = str(a).strip().lower(), str(b).strip().lower()
    if sa == sb: return True
    na, nb = parse_number_loose(a), parse_number_loose(b)
    if na is None or nb is None: return False
    # FinQA may represent percentages as either 23.6% or 0.236; parse handles that.
    return bool(np.isclose(na, nb, atol=atol, rtol=rtol))

# Source counts.
for split, expected in FINQA_EXPECTED.items():
    observed = len(finqa_splits[split])
    add_gate(f"FinQA {split} count", observed == expected, observed, expected)

# IDs.
add_gate("FinQA IDs globally unique", finqa_df["id"].nunique() == len(finqa_df),
         int(finqa_df["id"].nunique()), len(finqa_df))

required_cols = ["id", "question", "exe_ans", "program", "gold_inds", "table"]
for col in required_cols:
    missing = int(finqa_df[col].isna().sum()) if col in finqa_df else len(finqa_df)
    add_gate(f"FinQA required field present: {col}", missing == 0, missing, 0)

# The final step uses mixed official representations. Accept either the annotated
# human-facing answer or the executable numeric answer after normalization.
step_check = finqa_df[finqa_df["final_step_result"].notna()].copy()
step_check["final_matches_answer"] = step_check.apply(
    lambda r: loose_equivalent(r["final_step_result"], r["answer"]), axis=1
)
step_check["final_matches_exe_ans"] = step_check.apply(
    lambda r: loose_equivalent(r["final_step_result"], r["exe_ans"]), axis=1
)
step_check["final_consistent"] = step_check["final_matches_answer"] | step_check["final_matches_exe_ans"]
step_match_rate = float(step_check["final_consistent"].mean()) if len(step_check) else 0.0
answer_match_rate = float(step_check["final_matches_answer"].mean()) if len(step_check) else 0.0
exe_match_rate = float(step_check["final_matches_exe_ans"].mean()) if len(step_check) else 0.0
add_gate("FinQA final step matches an official answer representation", step_match_rate >= 0.99,
         round(step_match_rate, 5), ">=0.99",
         severity="WARNING",
         note="This source-annotation audit is non-blocking; the final step may correspond to qa.answer or qa.exe_ans, which are both preserved.")
print("Final-step/annotated-answer agreement:", round(answer_match_rate, 5))
print("Final-step/exe_ans agreement:", round(exe_match_rate, 5))

# Program vocabulary sanity check.
allowed_ops = {"add", "subtract", "multiply", "divide", "greater", "exp",
               "table_sum", "table_average", "table_max", "table_min"}
observed_ops = set()
for p in finqa_df["program"].dropna().astype(str):
    observed_ops.update(re.findall(r"([A-Za-z_][A-Za-z0-9_]*)\(", p))
unexpected_ops = sorted(observed_ops - allowed_ops)
add_gate("FinQA program operation vocabulary recognized", len(unexpected_ops) == 0,
         unexpected_ops, [])

# Exact full-input signatures across splits.
finqa_df["input_signature"] = finqa_df.apply(
    lambda r: input_signature(r["question"], r["context_text"], r["table_text"]), axis=1
)
sig_splits = finqa_df.groupby("input_signature")["split"].nunique()
cross_split_sig_count = int((sig_splits > 1).sum())
add_gate("FinQA no exact full-input leakage across splits", cross_split_sig_count == 0,
         cross_split_sig_count, 0)

print("Duplicate question strings (informational):", int(finqa_df["question"].duplicated().sum()))
print("Final-step/official-answer mismatches to inspect:", int((~step_check["final_consistent"]).sum()))
display(step_check.loc[~step_check["final_consistent"],
                       ["id", "split", "question", "answer", "exe_ans", "final_step_result", "program"]].head(20))

## 5. Official FinQA gold-program execution validation

The official FinQA repository defines execution accuracy using its `evaluate.py` program executor. Phase-2.1 downloads that exact evaluator, fingerprints it, tokenizes each gold program with the official tokenizer, executes it against the corresponding table, and compares the result with `qa.exe_ans`.

**Required research gate:** every canonical gold program must parse and reproduce its official execution answer. If this gate fails, downstream FinQA training/evaluation must not be trusted until the discrepancy is resolved.

In [ ]:
FINQA_OFFICIAL_EVAL_DIR = RAW_DIR / "finqa_official_evaluator"
FINQA_OFFICIAL_EVAL_DIR.mkdir(parents=True, exist_ok=True)
FINQA_OFFICIAL_EVALUATOR_PATH = FINQA_OFFICIAL_EVAL_DIR / "evaluate.py"
download_file(FINQA_EVALUATOR_URL, FINQA_OFFICIAL_EVALUATOR_PATH)

finqa_evaluator_manifest = {
    "url": FINQA_EVALUATOR_URL,
    "path": str(FINQA_OFFICIAL_EVALUATOR_PATH.resolve()),
    "sha256": sha256_file(FINQA_OFFICIAL_EVALUATOR_PATH),
}
print("Official evaluator manifest:", finqa_evaluator_manifest)

spec = importlib.util.spec_from_file_location(
    "finqa_official_evaluate", FINQA_OFFICIAL_EVALUATOR_PATH
)
finqa_official = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(finqa_official)

assert hasattr(finqa_official, "program_tokenization")
assert hasattr(finqa_official, "eval_program")
print("Loaded official FinQA program tokenizer and executor.")

In [ ]:
def validate_finqa_gold_execution(raw_path: Path, split: str) -> pd.DataFrame:
    raw = json.loads(raw_path.read_text(encoding="utf-8"))
    rows = []
    for item in tqdm(raw, desc=f"Official FinQA executor: {split}"):
        qa = item["qa"]
        tokens = finqa_official.program_tokenization(qa["program"])
        invalid_flag, executed = finqa_official.eval_program(tokens, item["table"])
        expected = qa["exe_ans"]
        rows.append({
            "id": str(item["id"]),
            "split": split,
            "invalid_flag": int(invalid_flag),
            "executed": executed,
            "expected_exe_ans": expected,
            # Use the official evaluator's equality rule: exact Python equality.
            "official_match": bool(invalid_flag == 0 and executed == expected),
            "program": qa["program"],
        })
    return pd.DataFrame(rows)

finqa_official_validation = {}
for split, path in finqa_raw_paths.items():
    df = validate_finqa_gold_execution(path, split)
    finqa_official_validation[split] = df

    invalid_n = int((df["invalid_flag"] != 0).sum())
    mismatch_n = int((~df["official_match"]).sum())
    add_gate(
        f"FinQA {split}: official gold programs execute successfully",
        invalid_n == 0,
        invalid_n,
        0,
    )
    add_gate(
        f"FinQA {split}: official execution equals qa.exe_ans",
        mismatch_n == 0,
        mismatch_n,
        0,
    )

finqa_exec_audit_df = pd.concat(finqa_official_validation.values(), ignore_index=True)
finqa_exec_audit_path = AUDIT_DIR / "finqa_official_gold_execution_audit.csv"
finqa_exec_audit_df.to_csv(finqa_exec_audit_path, index=False)

print("Official execution audit:", finqa_exec_audit_path)
display(
    finqa_exec_audit_df.groupby("split").agg(
        n=("id", "size"),
        invalid=("invalid_flag", lambda s: int((s != 0).sum())),
        exact_execution_match=("official_match", "sum"),
    )
)
if (~finqa_exec_audit_df["official_match"]).any():
    display(finqa_exec_audit_df.loc[~finqa_exec_audit_df["official_match"]].head(30))

## 6. FinQA canonical Phase-3/4 records

In [ ]:
FINQA_SYSTEM = (
    "You are a financial reasoning assistant. Use only the supplied financial evidence. "
    "Return a valid FinQA program and the final answer."
)


def make_finqa_record(row: pd.Series) -> Dict[str, Any]:
    input_text = (
        f"Financial text:\n{row['context_text']}\n\n"
        f"Financial table:\n{row['table_text']}\n\n"
        f"Question:\n{row['question']}"
    ).strip()
    target = f"PROGRAM: {row['program']}\nANSWER: {row['exe_ans']}"
    return {
        "record_id": f"finqa::{row['id']}",
        "source_id": row["id"],
        "dataset": "FinQA",
        "task": "financial_numerical_reasoning",
        "split": row["split"],
        "system": FINQA_SYSTEM,
        "instruction": "Solve the financial question from the supplied evidence.",
        "input": input_text,
        "target": target,
        "target_answer": row["exe_ans"],
        "target_label": None,
        "gold_program": row["program"],
        "gold_evidence": row["gold_inds"],
        "metadata": {
            "schema_version": SCHEMA_VERSION,
            "has_table": bool(row["table"]),
            "annotated_answer": row["answer"],
            "execution_answer": row["exe_ans"],
            "input_signature": row["input_signature"],
            "evaluation_only": row["split"] == "test",
        },
    }

finqa_records = {
    split: [make_finqa_record(r) for _, r in finqa_splits[split].assign(
        input_signature=finqa_splits[split].apply(
            lambda x: input_signature(x["question"], x["context_text"], x["table_text"]), axis=1
        )
    ).iterrows()]
    for split in ["train", "dev", "test"]
}

print({k: len(v) for k, v in finqa_records.items()})
print(json.dumps(finqa_records["train"][0], indent=2, ensure_ascii=False)[:2000])

# Part B — FiQA continuous financial sentiment
## 7. Load FiQA train / validation / test

FiQA is used in its native continuous sentiment formulation. Phase-2.1 does **not** discretize scores into arbitrary sentiment classes.

Canonical task:
- input: financial sentence + target entity + annotated aspect;
- target: continuous sentiment score in \([-1,1]\);
- Phase-4 primary metrics: MSE and \(R^2\), with MAE/Pearson as secondary diagnostics.

In [ ]:
fiqa_raw_splits: Dict[str, pd.DataFrame] = {}
for logical_split, hf_split in FIQA_SPLIT_MAP.items():
    df = hf_dataset_to_pandas(FIQA_HF_REPO, hf_split)
    fiqa_raw_splits[logical_split] = df
    print(logical_split, len(df), list(df.columns))

fiqa_source_manifest = {
    split: {
        "mode": "huggingface",
        "hf_repo": FIQA_HF_REPO,
        "hf_split": FIQA_SPLIT_MAP[split],
        "rows": int(len(df)),
        "sha256": sha256_text(
            canonical_json(
                sorted(
                    df.to_dict("records"),
                    key=lambda r: str(r.get("_id", r.get("id", ""))),
                )
            )
        ),
    }
    for split, df in fiqa_raw_splits.items()
}
display(pd.DataFrame(fiqa_source_manifest).T)

## 8. FiQA schema normalization and integrity gates

In [ ]:
FIQA_REQUIRED_COLUMNS = ["_id", "sentence", "target", "aspect", "score", "type"]

def normalize_fiqa_frame(df: pd.DataFrame, split: str) -> pd.DataFrame:
    missing = [c for c in FIQA_REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"FiQA {split}: missing required columns {missing}. Available={list(df.columns)}")

    out = pd.DataFrame()
    raw_ids = df["_id"].astype(str).tolist()
    # Deduplicate within split: append _0, _1, ... for repeated _id values.
    from collections import Counter as _Counter
    id_counts = _Counter(raw_ids)
    id_seen: Dict[str, int] = {}
    unique_ids = []
    for rid in raw_ids:
        if id_counts[rid] > 1:
            idx = id_seen.get(rid, 0)
            id_seen[rid] = idx + 1
            unique_ids.append(f"{rid}_{idx}")
        else:
            unique_ids.append(rid)
    out["id"] = unique_ids
    out["split"] = split
    out["sentence"] = df["sentence"].astype(str).str.strip()
    out["entity_target"] = df["target"].astype(str).str.strip()
    out["aspect"] = df["aspect"].astype(str).str.strip()
    out["score"] = pd.to_numeric(df["score"], errors="coerce")
    out["text_type"] = df["type"].astype(str).str.strip().str.lower()
    out["input_signature"] = out.apply(
        lambda r: input_signature(r["sentence"], r["entity_target"], r["aspect"]), axis=1
    )
    return out

fiqa_splits = {
    split: normalize_fiqa_frame(df, split)
    for split, df in fiqa_raw_splits.items()
}
fiqa_df = pd.concat(list(fiqa_splits.values()), ignore_index=True)

for split, expected in FIQA_EXPECTED.items():
    add_gate(
        f"FiQA {split} count",
        len(fiqa_splits[split]) == expected,
        int(len(fiqa_splits[split])),
        expected,
    )

add_gate(
    "FiQA IDs globally unique",
    fiqa_df["id"].nunique() == len(fiqa_df),
    int(fiqa_df["id"].nunique()),
    len(fiqa_df),
    severity="WARNING",
    note="If upstream IDs are reused across splits, Phase-2.1 record IDs still include the split to remain collision-safe.",
)
add_gate("FiQA sentence non-empty", bool(fiqa_df["sentence"].str.len().gt(0).all()))
add_gate("FiQA target entity non-empty", bool(fiqa_df["entity_target"].str.len().gt(0).all()))
add_gate("FiQA aspect non-empty", bool(fiqa_df["aspect"].str.len().gt(0).all()))
add_gate("FiQA score numeric", int(fiqa_df["score"].isna().sum()) == 0, int(fiqa_df["score"].isna().sum()), 0)
add_gate(
    "FiQA score bounded in [-1, 1]",
    bool(fiqa_df["score"].between(-1.0, 1.0, inclusive="both").all()),
    [float(fiqa_df["score"].min()), float(fiqa_df["score"].max())],
    [-1.0, 1.0],
)

fiqa_sig_cross_split = int(
    (fiqa_df.groupby("input_signature")["split"].nunique() > 1).sum()
)
add_gate(
    "FiQA no exact task-input leakage across splits",
    fiqa_sig_cross_split == 0,
    fiqa_sig_cross_split,
    0,
)

display(
    fiqa_df.groupby("split").agg(
        n=("id", "size"),
        score_mean=("score", "mean"),
        score_std=("score", "std"),
        score_min=("score", "min"),
        score_max=("score", "max"),
    ).round(4)
)
display(fiqa_df["text_type"].value_counts(dropna=False).rename("count"))

## 9. FiQA canonical Phase-3/4 records

The score is serialized without label discretization. Phase-3 can use the same generative SFT trainer as FinQA by treating the completion as a structured numeric target.

In [ ]:
FIQA_SYSTEM = (
    "You are a financial sentiment analysis assistant. Estimate the sentiment toward "
    "the specified financial target for the specified aspect. Return a numeric score "
    "from -1 (most negative) to +1 (most positive)."
)

def format_fiqa_score(x: float) -> str:
    return format(float(x), ".10g")

def make_fiqa_record(row: pd.Series) -> Dict[str, Any]:
    inp = (
        f"Financial text:\n{row['sentence']}\n\n"
        f"Target entity:\n{row['entity_target']}\n\n"
        f"Aspect:\n{row['aspect']}"
    ).strip()
    score = float(row["score"])
    return {
        "record_id": f"fiqa::{row['split']}::{row['id']}",
        "source_id": row["id"],
        "dataset": "FiQA",
        "task": "financial_sentiment_regression",
        "split": row["split"],
        "system": FIQA_SYSTEM,
        "instruction": "Return only `SENTIMENT_SCORE: <number>` with a value from -1 to +1.",
        "input": inp,
        "target": f"SENTIMENT_SCORE: {format_fiqa_score(score)}",
        "target_answer": score,
        "target_label": None,
        "gold_program": None,
        "gold_evidence": None,
        "metadata": {
            "schema_version": SCHEMA_VERSION,
            "entity_target": row["entity_target"],
            "aspect": row["aspect"],
            "text_type": row["text_type"],
            "sentiment_score": score,
            "input_signature": row["input_signature"],
            "evaluation_only": row["split"] == "test",
        },
    }

fiqa_records = {
    split: [make_fiqa_record(r) for _, r in df.iterrows()]
    for split, df in fiqa_splits.items()
}
print({k: len(v) for k, v in fiqa_records.items()})
print(json.dumps(fiqa_records["train"][0], indent=2, ensure_ascii=False))

# Part C — FinBen-FOMC protected external evaluation
## 10. Load and normalize FinBen-FOMC

The public repository exposes a single 496-example test split. Phase-2.1 therefore hard-codes the methodological rule that **zero FOMC examples may enter training or validation**.

In [ ]:
def load_finben_fomc_hf(repo_id: str, split: str) -> pd.DataFrame:
    from datasets import load_dataset
    ds = load_dataset(repo_id, split=split, token=HF_TOKEN or None)
    return ds.to_pandas()


if not HF_TOKEN:
    print("WARNING: HF_TOKEN is not set. The FinBen-FOMC dataset may be gated "
          "and require authentication.")

finben_fomc_raw = load_finben_fomc_hf(FINBEN_FOMC_HF_REPO, FINBEN_FOMC_SPLIT)
print("FinBen-FOMC rows:", len(finben_fomc_raw))
print("FinBen-FOMC columns:", list(finben_fomc_raw.columns))

finben_records_for_hash = sorted(
    finben_fomc_raw.to_dict("records"),
    key=lambda r: str(r.get("id")),
)
finben_source_manifest = {
    "mode": "huggingface",
    "hf_repo": FINBEN_FOMC_HF_REPO,
    "split": FINBEN_FOMC_SPLIT,
    "rows": int(len(finben_fomc_raw)),
    "sha256": sha256_text(canonical_json(finben_records_for_hash)),
}

In [ ]:
def first_existing(columns: Sequence[str], candidates: Sequence[str]) -> Optional[str]:
    lookup = {str(c).lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in lookup:
            return lookup[cand.lower()]
    return None


def normalize_fomc_label(x: Any) -> str:
    s = str(x).strip().lower()
    if s not in FINBEN_FOMC_LABELS:
        raise ValueError(f"Unknown FinBen-FOMC label: {x!r}")
    return s


def normalize_fomc_frame(df: pd.DataFrame, split: str) -> pd.DataFrame:
    cols = list(df.columns)
    text_col = first_existing(cols, ["text", "sentence", "premise"])
    answer_col = first_existing(cols, ["answer", "gold_label", "label"])
    query_col = first_existing(cols, ["query", "question", "prompt"])
    choices_col = first_existing(cols, ["choices"])
    gold_col = first_existing(cols, ["gold", "label_index", "answer_index"])
    id_col = first_existing(cols, ["id", "uid", "example_id", "index"])

    missing = [n for n, c in [("text", text_col), ("answer", answer_col)] if c is None]
    if missing:
        raise ValueError(f"FinBen-FOMC {split}: missing required columns {missing}. Available: {cols}")

    out = pd.DataFrame()
    out["id"] = df[id_col].astype(str) if id_col else [f"fomc-{i:05d}" for i in range(len(df))]
    out["split"] = split
    out["text"] = df[text_col].astype(str).str.strip()
    out["query"] = df[query_col].astype(str).str.strip() if query_col else ""
    out["answer"] = df[answer_col].map(normalize_fomc_label)
    if choices_col:
        out["choices"] = df[choices_col].apply(
            lambda c: [str(x) for x in c] if isinstance(c, (list, tuple)) else None
        )
    else:
        out["choices"] = None
    out["gold"] = df[gold_col].astype("Int64") if gold_col else None
    out["label"] = out["answer"]
    out["input_signature"] = out.apply(lambda r: input_signature(r["text"]), axis=1)
    return out


finben_splits: Dict[str, pd.DataFrame] = {}
FINBEN_AVAILABLE = bool(finben_fomc_raw is not None and len(finben_fomc_raw) > 0)

if FINBEN_AVAILABLE:
    finben_splits[FINBEN_FOMC_SPLIT] = normalize_fomc_frame(finben_fomc_raw, FINBEN_FOMC_SPLIT)
    finben_df = finben_splits[FINBEN_FOMC_SPLIT].copy()
    display(finben_df.head())

    add_gate("FinBen-FOMC source configured", True,
             observed=f"{FINBEN_FOMC_SPLIT} with {len(finben_df)} rows",
             expected=FINBEN_FOMC_HF_REPO)
    add_gate("FinBen-FOMC test count", len(finben_df) == FINBEN_FOMC_EXPECTED,
             int(len(finben_df)), FINBEN_FOMC_EXPECTED,
             note="Official split size reported on the Hugging Face repository.")
    add_gate("FinBen-FOMC IDs globally unique", finben_df["id"].nunique() == len(finben_df),
             int(finben_df["id"].nunique()), len(finben_df))
    add_gate("FinBen-FOMC text non-empty", bool(finben_df["text"].str.len().gt(0).all()))
    add_gate("FinBen-FOMC labels in expected vocabulary",
             set(finben_df["label"].unique()) == set(FINBEN_FOMC_LABELS),
             sorted(finben_df["label"].unique().tolist()), FINBEN_FOMC_LABELS)

    gold_total = 0
    gold_valid = 0
    for _, r in finben_df.iterrows():
        if r["choices"] is not None and r["gold"] is not None and not pd.isna(r["gold"]):
            gold_total += 1
            idx = int(r["gold"])
            if 0 <= idx < len(r["choices"]):
                gold_valid += int(str(r["choices"][idx]).strip().lower() == r["label"])
    if gold_total:
        add_gate("FinBen-FOMC choices[gold] consistent with answer",
                 gold_valid == gold_total, f"{gold_valid}/{gold_total}", gold_total)
    else:
        add_gate("FinBen-FOMC choices[gold] consistent with answer",
                 True, observed="no gold column", expected="choices[gold] == answer",
                 severity="WARNING", note="No gold index column found; the `answer` field is used as the label.")
else:
    add_gate("FinBen-FOMC source configured", False, observed="not found",
             expected=FINBEN_FOMC_HF_REPO,
             note="Phase-2 cannot pass until TheFinAI/finben-fomc is loadable with HF_TOKEN.")
    finben_df = pd.DataFrame()

In [ ]:
FINBEN_SYSTEM = (
    "You are an FOMC stance classifier. Classify each central-bank statement "
    "as HAWKISH, DOVISH, or NEUTRAL."
)


def make_finben_record(row: pd.Series) -> Dict[str, Any]:
    inp = f"Statement:\n{row['text']}"
    return {
        "record_id": f"finben-fomc::{row['id']}",
        "source_id": row["id"],
        "dataset": "FinBen-FOMC",
        "task": "fomc_stance",
        "split": row["split"],
        "system": FINBEN_SYSTEM,
        "instruction": "Return only HAWKISH, DOVISH, or NEUTRAL.",
        "input": inp,
        "target": row["label"],
        "target_answer": None,
        "target_label": row["label"],
        "gold_program": None,
        "gold_evidence": None,
        "metadata": {
            "schema_version": SCHEMA_VERSION,
            "query": row["query"],
            "choices": row["choices"],
            "gold_index": None if (row["gold"] is None or pd.isna(row["gold"])) else int(row["gold"]),
            "input_signature": row["input_signature"],
            "evaluation_only": True,
        },
    }


finben_records: Dict[str, List[Dict[str, Any]]] = {"train": [], "dev": [], "test": []}
if FINBEN_AVAILABLE:
    finben_records[FINBEN_FOMC_SPLIT] = [make_finben_record(r) for _, r in finben_df.iterrows()]
    print({k: len(v) for k, v in finben_records.items()})

In [ ]:
# Phase-2.1 hard isolation checks for FOMC.
add_gate(
    "FinBen-FOMC contributes zero training examples",
    len(finben_records.get("train", [])) == 0,
    len(finben_records.get("train", [])),
    0,
)
add_gate(
    "FinBen-FOMC contributes zero validation examples",
    len(finben_records.get("dev", [])) == 0,
    len(finben_records.get("dev", [])),
    0,
)
add_gate(
    "FinBen-FOMC all public records are evaluation-only",
    all(r["metadata"].get("evaluation_only") is True for r in finben_records.get("test", [])),
)

# Part D — FinTrust protected evaluation registry
## 11. Snapshot FinTrust without converting it into SFT data

In [ ]:
FINTRUST_RAW_DIR = RAW_DIR / "fintrust_hf"
FINTRUST_EVAL_CODE_DIR = RAW_DIR / "fintrust_eval_code"


def normalize_dimension_name(name: str) -> str:
    return name.lower().replace("-", "_").replace(" ", "_")


def snapshot_fintrust_hf() -> Path:
    from huggingface_hub import snapshot_download
    path = snapshot_download(
        repo_id=FINTRUST_HF_REPO,
        repo_type="dataset",
        local_dir=str(FINTRUST_RAW_DIR),
    )
    return Path(path)


def clone_or_update_fintrust_eval_repo() -> Optional[Path]:
    try:
        if (FINTRUST_EVAL_CODE_DIR / ".git").exists():
            subprocess.run(["git", "-C", str(FINTRUST_EVAL_CODE_DIR), "pull", "--ff-only"],
                           check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        elif not FINTRUST_EVAL_CODE_DIR.exists() or not any(FINTRUST_EVAL_CODE_DIR.iterdir()):
            subprocess.run(["git", "clone", "--depth", "1", FINTRUST_GITHUB, str(FINTRUST_EVAL_CODE_DIR)],
                           check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        return FINTRUST_EVAL_CODE_DIR
    except Exception as e:
        print("FinTrust evaluator repo clone/update warning:", repr(e))
        return None

FINTRUST_AVAILABLE = True
try:
    fintrust_snapshot = snapshot_fintrust_hf()
except Exception as e:
    FINTRUST_AVAILABLE = False
    fintrust_snapshot = FINTRUST_RAW_DIR
    print("FinTrust Hugging Face snapshot failed:", repr(e))

fintrust_eval_code = clone_or_update_fintrust_eval_repo()
print("FinTrust data snapshot:", fintrust_snapshot)
print("FinTrust evaluator code:", fintrust_eval_code)

## 12. Dimension-aware FinTrust manifest

In [ ]:
def infer_fintrust_dimension(path: Path, root: Path) -> Optional[str]:
    try:
        rel = path.relative_to(root)
    except ValueError:
        rel = path
    parts = [normalize_dimension_name(x) for x in rel.parts]
    joined = "_".join(parts)
    for dim in FINTRUST_DIMENSIONS:
        if dim in parts or dim in joined:
            return dim
    # tolerate hyphenated official directory naming
    if "knowledge_discovery" in joined:
        return "knowledge_discovery"
    return None


def classify_fintrust_file_role(path: Path) -> str:
    n = path.name.lower()
    if any(k in n for k in ["response", "output", "result", "prediction"]): return "generated_or_result"
    if any(k in n for k in ["prompt", "test", "sample", "dataset", "question", "data"]): return "candidate_eval_data"
    return "unknown"


def summarize_schema_from_records(records: List[Dict[str, Any]], limit: int = 200) -> List[str]:
    keys = set()
    for r in records[:limit]:
        keys.update(map(str, r.keys()))
    return sorted(keys)

TABULAR_EXT = {".csv", ".tsv", ".json", ".jsonl", ".parquet"}

fintrust_manifest_rows = []
if FINTRUST_AVAILABLE:
    data_files = [
        p for p in fintrust_snapshot.rglob("*")
        if p.is_file() and p.suffix.lower() in TABULAR_EXT and ".cache" not in p.parts
    ]
    for p in tqdm(data_files, desc="FinTrust files"):
        dim = infer_fintrust_dimension(p, fintrust_snapshot)
        try:
            df = load_tabular_file(p)
            schema = sorted(map(str, df.columns))
            n_rows = len(df)
            load_error = None
        except Exception as e:
            schema, n_rows, load_error = [], None, repr(e)
        fintrust_manifest_rows.append({
            "dimension": dim,
            "relative_path": str(p.relative_to(fintrust_snapshot)),
            "absolute_path": str(p.resolve()),
            "role": classify_fintrust_file_role(p),
            "rows": n_rows,
            "columns": schema,
            "sha256": sha256_file(p),
            "load_error": load_error,
            "evaluation_only": True,
        })

fintrust_manifest_df = pd.DataFrame(fintrust_manifest_rows)
if len(fintrust_manifest_df):
    display(fintrust_manifest_df[["dimension", "relative_path", "role", "rows", "load_error"]])
    display(fintrust_manifest_df.groupby("dimension", dropna=False)["rows"].sum(min_count=1).rename("raw_rows"))

## 13. FinTrust integrity and evaluator discovery

Unlike the older Phase-2 root-folder check, Phase-2.1 discovers evaluator/post-processing Python files **recursively** and maps candidate scripts back to trust dimensions from their paths. Missing dimension-specific scripts remain a warning; they are never replaced with a generic invented metric.

In [ ]:
if FINTRUST_AVAILABLE and len(fintrust_manifest_df):
    observed_dims = sorted(d for d in fintrust_manifest_df["dimension"].dropna().unique())
    add_gate(
        "FinTrust all seven dimensions represented",
        set(observed_dims) == set(FINTRUST_DIMENSIONS),
        observed_dims,
        sorted(FINTRUST_DIMENSIONS),
    )
    add_gate(
        "FinTrust data files load without parser errors",
        int(fintrust_manifest_df["load_error"].notna().sum()) == 0,
        int(fintrust_manifest_df["load_error"].notna().sum()),
        0,
    )
else:
    add_gate(
        "FinTrust dataset snapshot available",
        False,
        observed="unavailable",
        expected=FINTRUST_HF_REPO,
    )

fintrust_eval_candidates: Dict[str, List[str]] = {d: [] for d in FINTRUST_DIMENSIONS}
if fintrust_eval_code and fintrust_eval_code.exists():
    py_files = list(fintrust_eval_code.rglob("*.py"))
    for p in py_files:
        rel = str(p.relative_to(fintrust_eval_code)).replace("\\", "/")
        norm = normalize_dimension_name(rel)
        if any(k in p.name.lower() for k in ["postprocess", "evaluate", "metric", "score"]):
            for dim in FINTRUST_DIMENSIONS:
                aliases = [dim, dim.replace("_", "-")]
                if any(normalize_dimension_name(a) in norm for a in aliases):
                    fintrust_eval_candidates[dim].append(rel)

    discovered_dims = sorted(d for d, paths in fintrust_eval_candidates.items() if paths)
    add_gate(
        "FinTrust evaluator/postprocess candidates discovered for all dimensions",
        set(discovered_dims) == set(FINTRUST_DIMENSIONS),
        discovered_dims,
        FINTRUST_DIMENSIONS,
        severity="WARNING",
        note=(
            "Keep official task-specific evaluation. Missing candidates must be resolved in Phase-4; "
            "do not replace them with a generic metric."
        ),
    )
else:
    add_gate(
        "FinTrust official evaluator repository available",
        False,
        observed="git clone unavailable",
        expected=FINTRUST_GITHUB,
        severity="WARNING",
    )

display(pd.DataFrame([
    {"dimension": d, "candidate_scripts": paths}
    for d, paths in fintrust_eval_candidates.items()
]))

# Part E — Unified Phase-3/4 contract
## 14. Build training, validation, and protected evaluation pools

In [ ]:
# Only FinQA and FiQA are allowed in Phase-3 SFT.
train_records = list(finqa_records["train"]) + list(fiqa_records["train"])
validation_records = list(finqa_records["dev"]) + list(fiqa_records["valid"])

# Protected evaluation.
finqa_test_records = list(finqa_records["test"])
fiqa_test_records = list(fiqa_records["test"])
finben_test_records = list(finben_records["test"])

protected_eval_records = finqa_test_records + fiqa_test_records + finben_test_records

print("Train:", len(train_records), Counter(r["dataset"] for r in train_records))
print("Validation:", len(validation_records), Counter(r["dataset"] for r in validation_records))
print("FinQA test:", len(finqa_test_records))
print("FiQA test:", len(fiqa_test_records))
print("FinBen-FOMC test:", len(finben_test_records))

assert_record_contract(train_records, "Train")
assert_record_contract(validation_records, "Validation")
assert_record_contract(protected_eval_records, "Protected evaluation")

add_gate(
    "Training datasets exactly FinQA + FiQA",
    set(r["dataset"] for r in train_records) == {"FinQA", "FiQA"},
    sorted(set(r["dataset"] for r in train_records)),
    ["FinQA", "FiQA"],
)
add_gate(
    "No evaluation_only records in training",
    all(not r["metadata"].get("evaluation_only", False) for r in train_records),
)
add_gate(
    "No evaluation_only records in validation",
    all(not r["metadata"].get("evaluation_only", False) for r in validation_records),
)
add_gate(
    "No FinTrust records in training/validation",
    all(r["dataset"] != "FinTrust" for r in train_records + validation_records),
)
add_gate(
    "No FinBen-FOMC records in training/validation",
    all(r["dataset"] != "FinBen-FOMC" for r in train_records + validation_records),
)

## 15. Mandatory exact leakage audit

The audit uses normalized task inputs, not just IDs. Every protected test pool is independently checked against training.

In [ ]:
def record_input_sig(r: Mapping[str, Any]) -> str:
    return r.get("metadata", {}).get("input_signature") or input_signature(r.get("input", ""))

def sig_set(records: Sequence[Mapping[str, Any]]) -> set:
    return {record_input_sig(r) for r in records}

train_sigs = sig_set(train_records)
val_sigs = sig_set(validation_records)
finqa_test_sigs = sig_set(finqa_test_records)
fiqa_test_sigs = sig_set(fiqa_test_records)
finben_test_sigs = sig_set(finben_test_records)

leak_checks = {
    "train_vs_validation": len(train_sigs & val_sigs),
    "train_vs_finqa_test": len(train_sigs & finqa_test_sigs),
    "train_vs_fiqa_test": len(train_sigs & fiqa_test_sigs),
    "train_vs_finben_fomc_test": len(train_sigs & finben_test_sigs),
}
for name, n in leak_checks.items():
    add_gate(f"Exact input leakage {name}", n == 0, n, 0)

display(pd.DataFrame(
    [{"comparison": k, "exact_overlap": v} for k, v in leak_checks.items()]
))

## 16. Mandatory near-duplicate train↔protected-test audit

Exact hashes cannot detect lightly edited duplicates. Phase-2.1 therefore runs MinHash-LSH over word shingles and verifies every candidate using exact Jaccard similarity.

- Threshold: `NEAR_DUP_THRESHOLD` (default 0.90).
- This is an **audit**, not an automatic deletion rule.
- Candidates are exported for manual review because legitimately repeated financial boilerplate can produce high similarity.
- Exact duplicates remain hard failures through the preceding hash audit.

In [ ]:
def word_shingles(text: Any, n: int = WORD_SHINGLE_N) -> set:
    words = re.findall(r"\w+(?:[.%$-]\w+)?", normalized_text(text))
    if not words:
        return {""}
    if len(words) < n:
        return {" ".join(words)}
    return {" ".join(words[i:i+n]) for i in range(len(words) - n + 1)}

def duplicate_audit_text(record: Mapping[str, Any]) -> str:
    return str(record.get("input", ""))

def jaccard_sets(a: set, b: set) -> float:
    union = a | b
    return 1.0 if not union else len(a & b) / len(union)

def run_minhash_audit(
    train: Sequence[Mapping[str, Any]],
    eval_records: Sequence[Mapping[str, Any]],
    threshold: float = NEAR_DUP_THRESHOLD,
) -> pd.DataFrame:
    from datasketch import MinHash, MinHashLSH

    def make_mh(shingles: set) -> Any:
        m = MinHash(num_perm=MINHASH_PERMUTATIONS, seed=SEED)
        for token in sorted(shingles):
            m.update(token.encode("utf-8"))
        return m

    lsh = MinHashLSH(threshold=threshold, num_perm=MINHASH_PERMUTATIONS)
    train_cache = {}

    for i, r in enumerate(tqdm(train, desc="Near-dup: index train")):
        shingles = word_shingles(duplicate_audit_text(r))
        m = make_mh(shingles)
        key = f"tr::{i}"
        lsh.insert(key, m)
        train_cache[key] = (r, shingles)

    hits = []
    for er in tqdm(eval_records, desc="Near-dup: audit protected eval"):
        e_shingles = word_shingles(duplicate_audit_text(er))
        e_mh = make_mh(e_shingles)
        for key in lsh.query(e_mh):
            tr, t_shingles = train_cache[key]
            jac = jaccard_sets(t_shingles, e_shingles)
            if jac >= threshold:
                hits.append({
                    "train_id": tr["record_id"],
                    "eval_id": er["record_id"],
                    "train_dataset": tr["dataset"],
                    "eval_dataset": er["dataset"],
                    "jaccard_word_shingles": float(jac),
                    "train_input_preview": duplicate_audit_text(tr)[:400],
                    "eval_input_preview": duplicate_audit_text(er)[:400],
                })

    if not hits:
        return pd.DataFrame(columns=[
            "train_id", "eval_id", "train_dataset", "eval_dataset",
            "jaccard_word_shingles", "train_input_preview", "eval_input_preview",
        ])

    out = pd.DataFrame(hits).drop_duplicates(["train_id", "eval_id"])
    return out.sort_values("jaccard_word_shingles", ascending=False).reset_index(drop=True)

if RUN_NEAR_DUP_AUDIT:
    near_dup_df = run_minhash_audit(train_records, protected_eval_records)
else:
    near_dup_df = pd.DataFrame()

near_dup_path = AUDIT_DIR / "near_duplicate_train_vs_protected_test.csv"
near_dup_df.to_csv(near_dup_path, index=False)

add_gate(
    f"Near-duplicate audit has no candidates >= {NEAR_DUP_THRESHOLD:.2f}",
    len(near_dup_df) == 0,
    int(len(near_dup_df)),
    0,
    severity="WARNING",
    note=(
        "Near-duplicate candidates require manual inspection; they are not deleted automatically. "
        "Exact duplicates are already hard failures."
    ),
)
print("Near-duplicate audit:", near_dup_path)
display(near_dup_df.head(50))

## 17. Protected-record ID isolation

In [ ]:
train_ids = {r["record_id"] for r in train_records}
val_ids = {r["record_id"] for r in validation_records}
protected_ids = {r["record_id"] for r in protected_eval_records}

add_gate("Train record IDs unique", len(train_ids) == len(train_records), len(train_ids), len(train_records))
add_gate("Validation record IDs unique", len(val_ids) == len(validation_records), len(val_ids), len(validation_records))
add_gate("Protected test record IDs unique", len(protected_ids) == len(protected_eval_records), len(protected_ids), len(protected_eval_records))
add_gate("No protected record IDs in training", len(train_ids & protected_ids) == 0, len(train_ids & protected_ids), 0)
add_gate("No protected record IDs in validation", len(val_ids & protected_ids) == 0, len(val_ids & protected_ids), 0)
add_gate("Train/validation record IDs disjoint", len(train_ids & val_ids) == 0, len(train_ids & val_ids), 0)

# Part F — Stable downstream artifacts
## 18. Export Phase-3 / Phase-4 datasets

These exported files are the **only supported downstream dataset interface**.

In [ ]:
train_path = EXPORT_DIR / "train.jsonl"
val_path = EXPORT_DIR / "validation.jsonl"

finqa_test_path = EVAL_DIR / "finqa_test.jsonl"
fiqa_test_path = EVAL_DIR / "fiqa_test.jsonl"
finben_fomc_test_path = EVAL_DIR / "finben_fomc_test.jsonl"

write_jsonl(train_records, train_path)
write_jsonl(validation_records, val_path)
write_jsonl(finqa_test_records, finqa_test_path)
write_jsonl(fiqa_test_records, fiqa_test_path)
write_jsonl(finben_test_records, finben_fomc_test_path)

fintrust_registry = {
    "dataset": "FinTrust",
    "evaluation_only": True,
    "hf_repo": FINTRUST_HF_REPO,
    "github_repo": FINTRUST_GITHUB,
    "dimensions": FINTRUST_DIMENSIONS,
    "data_snapshot": str(fintrust_snapshot.resolve()) if FINTRUST_AVAILABLE else None,
    "official_eval_code": str(fintrust_eval_code.resolve()) if fintrust_eval_code else None,
    "evaluator_candidates_by_dimension": fintrust_eval_candidates,
    "files": fintrust_manifest_df.to_dict(orient="records") if len(fintrust_manifest_df) else [],
    "note": (
        "Use official dimension-specific evaluation/postprocessing. "
        "Never convert FinTrust benchmark records into SFT or calibration data."
    ),
}
fintrust_registry_path = EVAL_DIR / "fintrust_registry.json"
save_json(fintrust_registry, fintrust_registry_path)

print("Exported:")
for p in [
    train_path, val_path, finqa_test_path, fiqa_test_path,
    finben_fomc_test_path, fintrust_registry_path,
]:
    print(" -", p)

## 19. Dataset registry, source fingerprints, and Phase-3/4 configs

In [ ]:
def artifact_entry(
    path: Path,
    role: str,
    dataset: str | List[str],
    task: str | List[str],
):
    return {
        "path": str(path.resolve()),
        "role": role,
        "dataset": dataset,
        "task": task,
        "sha256": sha256_file(path),
        "records": sum(1 for _ in path.open("r", encoding="utf-8")) if path.suffix == ".jsonl" else None,
    }

source_fingerprints = {
    "FinQA": finqa_source_manifest,
    "FinQA_official_evaluator": finqa_evaluator_manifest,
    "FiQA": fiqa_source_manifest,
    "FinBen-FOMC": finben_source_manifest,
    "FinTrust": {
        "hf_repo": FINTRUST_HF_REPO,
        "github_repo": FINTRUST_GITHUB,
        "dimensions": FINTRUST_DIMENSIONS,
        "file_sha256": {
            r["relative_path"]: r["sha256"]
            for r in fintrust_manifest_df.to_dict(orient="records")
        } if len(fintrust_manifest_df) else {},
    },
}
source_fingerprints_path = MANIFEST_DIR / "source_fingerprints.json"
save_json(source_fingerprints, source_fingerprints_path)

registry = {
    "schema_version": SCHEMA_VERSION,
    "created_unix": int(time.time()),
    "seed": SEED,
    "parent_phase2": phase2_parent_provenance,
    "training_policy": {
        "allowed_datasets": ["FinQA", "FiQA"],
        "forbidden_datasets": ["FinBen-FOMC", "FinTrust"],
        "reason": (
            "FinBen-FOMC is public test-only; FinTrust is trust evaluation-only. "
            "Neither may be used for SFT, validation, calibration, threshold tuning, or few-shot example selection."
        ),
    },
    "sampling_policy": {
        "records_are_exported_unweighted": True,
        "recommended_phase3_default": "dataset_balanced",
        "recommended_dataset_probabilities": {"FinQA": 0.5, "FiQA": 0.5},
        "note": (
            "Phase-3 should log the actual sampler. Also report a natural/proportional mix baseline "
            "if multi-task sampling is part of the claimed method."
        ),
    },
    "threshold_policy": {
        "calibration_and_abstention_threshold_selection": "validation only",
        "never_tune_on": ["FinQA test", "FiQA test", "FinBen-FOMC", "FinTrust"],
    },
    "artifacts": {
        "train": artifact_entry(
            train_path, "sft_train",
            ["FinQA", "FiQA"],
            ["financial_numerical_reasoning", "financial_sentiment_regression"],
        ),
        "validation": artifact_entry(
            val_path, "validation_and_calibration",
            ["FinQA", "FiQA"],
            ["financial_numerical_reasoning", "financial_sentiment_regression"],
        ),
        "finqa_test": artifact_entry(
            finqa_test_path, "protected_test", "FinQA", "financial_numerical_reasoning"
        ),
        "fiqa_test": artifact_entry(
            fiqa_test_path, "protected_test", "FiQA", "financial_sentiment_regression"
        ),
        "finben_fomc_test": artifact_entry(
            finben_fomc_test_path, "protected_external_test", "FinBen-FOMC", "fomc_stance"
        ),
        "fintrust_registry": {
            "path": str(fintrust_registry_path.resolve()),
            "role": "protected_trust_evaluation",
            "dataset": "FinTrust",
            "sha256": sha256_file(fintrust_registry_path),
        },
        "finqa_gold_execution_audit": {
            "path": str(finqa_exec_audit_path.resolve()),
            "role": "research_integrity_audit",
            "sha256": sha256_file(finqa_exec_audit_path),
        },
        "near_duplicate_audit": {
            "path": str(near_dup_path.resolve()),
            "role": "research_integrity_audit",
            "sha256": sha256_file(near_dup_path),
        },
    },
    "source_fingerprints": str(source_fingerprints_path.resolve()),
}
registry_path = MANIFEST_DIR / "dataset_registry.json"
save_json(registry, registry_path)

phase3_config = {
    "schema_version": SCHEMA_VERSION,
    "data_registry": str(registry_path.resolve()),
    "train_artifact": "train",
    "validation_artifact": "validation",
    "allowed_training_datasets": ["FinQA", "FiQA"],
    "forbidden_training_datasets": ["FinBen-FOMC", "FinTrust"],
    "default_sampling": {
        "strategy": "dataset_balanced",
        "dataset_probabilities": {"FinQA": 0.5, "FiQA": 0.5},
    },
    "target_format": {
        "FinQA": "PROGRAM: <program>\\nANSWER: <execution_answer>",
        "FiQA": "SENTIMENT_SCORE: <float in [-1,1]>",
    },
    "task_metrics_for_model_selection": {
        "FinQA": ["execution_accuracy", "program_accuracy"],
        "FiQA": ["mse", "r2"],
    },
    "important": (
        "Do not load protected tests during training except in a final evaluation process "
        "after model/configuration selection is frozen."
    ),
}
phase3_config_path = MANIFEST_DIR / "phase3_data_config.json"
save_json(phase3_config, phase3_config_path)

phase4_config = {
    "schema_version": SCHEMA_VERSION,
    "data_registry": str(registry_path.resolve()),
    "calibration_split": "validation",
    "protected_tests": ["finqa_test", "fiqa_test", "finben_fomc_test", "fintrust_registry"],
    "recommended_metrics": {
        "FinQA": [
            "official_execution_accuracy", "official_program_accuracy",
            "answer_accuracy", "parse_success", "answer_program_consistency",
        ],
        "FiQA": ["mse", "r2", "mae", "pearson_r"],
        "FinBen-FOMC": ["accuracy", "macro_f1", "per_class_f1"],
        "Calibration": ["task_appropriate_calibration_error", "brier_or_nll_when_probabilities_exist"],
        "SelectivePrediction": ["coverage", "selective_risk", "AURC"],
        "FinTrust": ["official_dimension_specific_metrics"],
        "Efficiency": ["trainable_parameters", "peak_vram", "latency", "tokens_per_second"],
    },
    "official_evaluators": {
        "FinQA_local": str(FINQA_OFFICIAL_EVALUATOR_PATH.resolve()),
        "FinQA_sha256": finqa_evaluator_manifest["sha256"],
        "FinTrust_local": str(fintrust_eval_code.resolve()) if fintrust_eval_code else None,
    },
    "evaluation_rules": {
        "FinQA": "Use the official executor for program/execution accuracy.",
        "FiQA": "Parse a numeric score, clip only if the experimental protocol explicitly says so, and report the rule.",
        "FinBen-FOMC": "Test-only external generalization; never tune on this split.",
        "FinTrust": "Use task/dimension-specific official evaluation; never invent a single generic trust score without justification.",
    },
}
phase4_config_path = MANIFEST_DIR / "phase4_eval_config.json"
save_json(phase4_config, phase4_config_path)

print("Registry:", registry_path)
print("Phase-3 config:", phase3_config_path)
print("Phase-4 config:", phase4_config_path)

## 20. Export a small importable interface for Phase-3/4

Phase-3/4 should import this module instead of copying Phase-2.1 data-loading code.

In [ ]:
interface_code = r"""from __future__ import annotations
import json
from pathlib import Path


def _read_jsonl(path):
    path = Path(path)
    out = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                out.append(json.loads(line))
    return out


def _read_json(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)


def load_registry(registry_path):
    return _read_json(registry_path)


def resolve_artifact(registry_path, key):
    reg = load_registry(registry_path)
    if key not in reg["artifacts"]:
        raise KeyError(f"Unknown artifact {key!r}. Available={sorted(reg['artifacts'])}")
    return reg["artifacts"][key]


def load_artifact(registry_path, key):
    entry = resolve_artifact(registry_path, key)
    path = Path(entry["path"])
    if path.suffix == ".jsonl":
        return _read_jsonl(path)
    if path.suffix == ".json":
        return _read_json(path)
    return path


def load_train(registry_path):
    records = load_artifact(registry_path, "train")
    assert all(not r.get("metadata", {}).get("evaluation_only", False) for r in records)
    return records


def load_validation(registry_path):
    records = load_artifact(registry_path, "validation")
    assert all(not r.get("metadata", {}).get("evaluation_only", False) for r in records)
    return records


def load_finqa_test(registry_path):
    return load_artifact(registry_path, "finqa_test")


def load_fiqa_test(registry_path):
    return load_artifact(registry_path, "fiqa_test")


def load_finben_fomc_test(registry_path):
    return load_artifact(registry_path, "finben_fomc_test")


def load_fintrust_registry(registry_path):
    return load_artifact(registry_path, "fintrust_registry")


def filter_records(records, dataset=None, task=None):
    out = list(records)
    if dataset is not None:
        out = [r for r in out if r.get("dataset") == dataset]
    if task is not None:
        out = [r for r in out if r.get("task") == task]
    return out


def dataset_counts(records):
    counts = {}
    for r in records:
        counts[r["dataset"]] = counts.get(r["dataset"], 0) + 1
    return counts
"""
interface_path = MODULE_DIR / "phase2_1_interface.py"
interface_path.write_text(interface_code, encoding="utf-8")
print("Saved:", interface_path)

# Part G — Final artifact-level validation
## 21. Re-open exports and validate invariants

In [ ]:
export_train = read_jsonl(train_path)
export_val = read_jsonl(val_path)
export_finqa_test = read_jsonl(finqa_test_path)
export_fiqa_test = read_jsonl(fiqa_test_path)
export_finben_test = read_jsonl(finben_fomc_test_path)

add_gate(
    "Exported train count preserved",
    len(export_train) == len(train_records),
    len(export_train),
    len(train_records),
)
add_gate(
    "Exported validation count preserved",
    len(export_val) == len(validation_records),
    len(export_val),
    len(validation_records),
)
add_gate(
    "FinQA test marked evaluation-only",
    all(r["metadata"].get("evaluation_only") for r in export_finqa_test),
)
add_gate(
    "FiQA test marked evaluation-only",
    all(r["metadata"].get("evaluation_only") for r in export_fiqa_test),
)
add_gate(
    "FinBen-FOMC test marked evaluation-only",
    all(r["metadata"].get("evaluation_only") for r in export_finben_test),
)
add_gate("FinTrust registry marked evaluation-only", fintrust_registry.get("evaluation_only") is True)

export_train_ids = {r["record_id"] for r in export_train}
export_protected_ids = {
    r["record_id"]
    for r in export_finqa_test + export_fiqa_test + export_finben_test
}
add_gate(
    "No exported protected test IDs in exported training",
    len(export_train_ids & export_protected_ids) == 0,
    len(export_train_ids & export_protected_ids),
    0,
)

# Verify hashes in the registry against current bytes.
hash_mismatches = []
for key, entry in registry["artifacts"].items():
    if "path" in entry and "sha256" in entry:
        p = Path(entry["path"])
        if p.exists() and sha256_file(p) != entry["sha256"]:
            hash_mismatches.append(key)
add_gate("Export artifact hashes reproduce registry", len(hash_mismatches) == 0, hash_mismatches, [])

## 22. Phase-2.1 PASS/FAIL and Phase-3/4 readiness report

Only `ERROR`-severity gate failures block downstream modeling. `WARNING` gates remain visible in the manifest and should be discussed/inspected before the final paper freeze.

In [ ]:
report_rows = [asdict(g) for g in GATES]
report_df = pd.DataFrame(report_rows)
display(report_df)

hard_failures = [g for g in GATES if g.severity == "ERROR" and not g.passed]
warnings = [g for g in GATES if g.severity == "WARNING" and not g.passed]
phase2_1_pass = len(hard_failures) == 0

ready_for_phase3 = (
    phase2_1_pass
    and set(r["dataset"] for r in export_train) == {"FinQA", "FiQA"}
    and all(not r["metadata"].get("evaluation_only", False) for r in export_train)
)
ready_for_phase4 = (
    phase2_1_pass
    and len(export_finqa_test) == FINQA_EXPECTED["test"]
    and len(export_fiqa_test) == FIQA_EXPECTED["test"]
    and len(export_finben_test) == FINBEN_FOMC_EXPECTED
    and fintrust_registry.get("evaluation_only") is True
)

validation_report = {
    "schema_version": SCHEMA_VERSION,
    "phase2_1_pass": phase2_1_pass,
    "ready_for_phase3": ready_for_phase3,
    "ready_for_phase4": ready_for_phase4,
    "hard_failures": [asdict(g) for g in hard_failures],
    "warnings": [asdict(g) for g in warnings],
    "gates": report_rows,
    "artifact_registry": str(registry_path.resolve()),
    "phase3_config": str(phase3_config_path.resolve()),
    "phase4_config": str(phase4_config_path.resolve()),
    "interface_module": str(interface_path.resolve()),
    "near_duplicate_audit": str(near_dup_path.resolve()),
    "finqa_official_execution_audit": str(finqa_exec_audit_path.resolve()),
}
validation_report_path = MANIFEST_DIR / "phase2_1_validation_report.json"
save_json(validation_report, validation_report_path)

print("=" * 88)
print("PHASE-2.1 STATUS:", "PASS" if phase2_1_pass else "FAIL")
print("READY FOR PHASE-3:", ready_for_phase3)
print("READY FOR PHASE-4:", ready_for_phase4)
print("Hard failures:", len(hard_failures))
print("Unresolved warnings:", len(warnings))
for g in hard_failures:
    print("ERROR:", g.name, "| observed:", g.observed, "| expected:", g.expected)
for g in warnings:
    print("WARNING:", g.name, "| observed:", g.observed, "| expected:", g.expected)
print("Validation report:", validation_report_path)
print("=" * 88)

## 23. Phase-3 / Phase-4 handoff contract

### Phase-3 — Baselines and QLoRA

Use only:

- `artifacts/phase2_1/datasets/train.jsonl`
- `artifacts/phase2_1/datasets/validation.jsonl`
- `artifacts/phase2_1/manifests/phase3_data_config.json`
- `artifacts/phase2_1/python/phase2_1_interface.py`

The training pool contains **FinQA + FiQA only**. The default config proposes dataset-balanced sampling but the actual sampler must be logged in every experiment.

Recommended Phase-3 experiment matrix:

1. Base SLM zero-shot on FinQA / FiQA validation.
2. Few-shot baseline using **training examples only**.
3. FinQA-only QLoRA.
4. FiQA-only QLoRA.
5. Joint FinQA + FiQA QLoRA with explicitly logged sampling.
6. Freeze model/configuration selection before protected-test evaluation.

### Phase-4 — TrustFin-SLM methodology

Protected evaluation artifacts:

- `eval/finqa_test.jsonl`
- `eval/fiqa_test.jsonl`
- `eval/finben_fomc_test.jsonl`
- `eval/fintrust_registry.json`
- `manifests/phase4_eval_config.json`

Research rules:

- FinQA program/execution metrics must use the **official FinQA executor** fingerprinted in this notebook.
- FiQA remains continuous regression; do not silently discretize labels.
- FinBen-FOMC is external test-only generalization.
- FinTrust remains untouched trust evaluation.
- Calibration/abstention thresholds are selected on validation only.
- Near-duplicate candidates, if any, are inspected and documented rather than silently removed.